# Extended Summary Statistics for Exploratory Data Analysis

**Source material:** *Exploratory Data Analysis with Python Cookbook* (Packt) – Chapter 1: Generating Summary Statistics

This notebook consolidates and **extends** the nine original recipes:
1. Mean  2. Median  3. Mode  4. Variance  5. Standard Deviation  
6. Range  7. Percentiles  8. Quartiles  9. Interquartile Range (IQR)

**Enhancements included:**
- Reusable function module (`summary_statistics.py`)
- Side-by-side numpy vs pandas comparison (ddof differences)
- Full descriptive summary in one call
- Group-by analysis (by continent / location)
- Visualisations (histograms, boxplots)
- Skewness & kurtosis
- Template pattern for any numeric column


## 1. Import libraries and the reusable module

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Make the scripts folder importable
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))

from summary_statistics import (
    compute_mean, compute_median, compute_mode,
    compute_variance, compute_std, compute_range,
    compute_percentile, compute_quartile, compute_iqr,
    full_summary, summary_by_group, print_summary_report
)

print('Libraries and module loaded successfully.')


## 2. Load and inspect the dataset

We use a sample of COVID-19 daily new cases (structure matches the original cookbook data).

In [ ]:
data_path = ROOT / 'data' / 'covid-data.csv'
covid_data = pd.read_csv(data_path)

# Keep the same columns as the original notebooks
covid_data = covid_data[['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases']]

print('Shape:', covid_data.shape)
display(covid_data.head())
print('\nDtypes:')
print(covid_data.dtypes)


## 3. Mean

The arithmetic mean is the average. It is sensitive to outliers.

In [ ]:
# Original-style calculation
data_mean_np = np.mean(covid_data['new_cases'])
data_mean_pd = covid_data['new_cases'].mean()

print(f'numpy  mean: {data_mean_np:,.4f}')
print(f'pandas mean: {data_mean_pd:,.4f}')
print(f'function   : {compute_mean(covid_data["new_cases"]):,.4f}')


## 4. Median

The median is the middle value when data are sorted. More robust to outliers than the mean.

In [ ]:
print(f'numpy  median: {np.median(covid_data["new_cases"]):,.4f}')
print(f'pandas median: {covid_data["new_cases"].median():,.4f}')
print(f'function     : {compute_median(covid_data["new_cases"]):,.4f}')


## 5. Mode

The mode is the most frequent value. Useful for both numeric and categorical columns.

In [ ]:
mode_new = compute_mode(covid_data['new_cases'])
print('Mode of new_cases:', mode_new)

mode_cont = compute_mode(covid_data['continent'])
print('Mode of continent:', mode_cont)


## 6. Variance

Variance measures the average squared deviation from the mean.

**Important:** `numpy.var` defaults to `ddof=0` (population), while `pandas.Series.var` defaults to `ddof=1` (sample).

In [ ]:
print('numpy  var (ddof=0):', np.var(covid_data['new_cases']))
print('pandas var (ddof=1):', covid_data['new_cases'].var())  # default ddof=1
print()
print('Via function (ddof=0):', compute_variance(covid_data['new_cases'], ddof=0))
print('Via function (ddof=1):', compute_variance(covid_data['new_cases'], ddof=1))


## 7. Standard Deviation

Standard deviation is the square root of variance and is expressed in the same units as the data.

In [ ]:
print('numpy  std (ddof=0):', np.std(covid_data['new_cases']))
print('pandas std (ddof=1):', covid_data['new_cases'].std())
print()
print(compute_std(covid_data['new_cases'], ddof=0))
print(compute_std(covid_data['new_cases'], ddof=1))


## 8. Range

Range = maximum − minimum. Simple but sensitive to extreme values.

In [ ]:
rng = compute_range(covid_data['new_cases'])
print(rng)
print(f"Range = {rng['max']:,.0f} - {rng['min']:,.0f} = {rng['range']:,.0f}")


## 9. Percentiles

A percentile indicates the value below which a given percentage of observations fall.

In [ ]:
for p in [10, 25, 50, 60, 75, 90, 95, 99]:
    val = compute_percentile(covid_data['new_cases'], p)
    print(f'{p:2d}th percentile: {val:>12,.2f}')


## 10. Quartiles

Quartiles divide the data into four equal parts (Q1=25 %, Q2=50 %, Q3=75 %).

In [ ]:
for q, label in [(0.25, 'Q1'), (0.50, 'Q2'), (0.75, 'Q3')]:
    print(f'{label} ({q}): {compute_quartile(covid_data["new_cases"], q):,.2f}')


## 11. Interquartile Range (IQR)

IQR = Q3 − Q1. It measures the spread of the middle 50 % of the data and is robust to outliers.

In [ ]:
iqr_info = compute_iqr(covid_data['new_cases'])
print(iqr_info)

# Also show the default (no explicit interpolation)
print('scipy.stats.iqr (default):', stats.iqr(covid_data['new_cases']))


## 12. Full Summary (one-call convenience function)

In [ ]:
summary = full_summary(covid_data['new_cases'])
print_summary_report(summary, 'new_cases – Full Summary')


## 13. Enhancement: Group-by analysis

Compute the same statistics for each continent and each location.

In [ ]:
print('=== By Continent ===')
by_continent = summary_by_group(covid_data, 'new_cases', 'continent')
display(by_continent)

print('\n=== By Location ===')
by_location = summary_by_group(covid_data, 'new_cases', 'location')
display(by_location)


## 14. Enhancement: Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(covid_data['new_cases'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(summary['mean'], color='red', linestyle='--', label=f"Mean={summary['mean']:.0f}")
axes[0].axvline(summary['median'], color='green', linestyle='--', label=f"Median={summary['median']:.0f}")
axes[0].set_title('Distribution of new_cases')
axes[0].set_xlabel('new_cases')
axes[0].legend()

# Boxplot (log scale helps with heavy right tail)
axes[1].boxplot(covid_data['new_cases'], vert=True)
axes[1].set_yscale('log')
axes[1].set_title('Boxplot (log scale)')

# Boxplot by continent
covid_data.boxplot(column='new_cases', by='continent', ax=axes[2])
axes[2].set_yscale('log')
axes[2].set_title('new_cases by Continent')
plt.suptitle('')

plt.tight_layout()
plt.show()


## 15. How to reuse this work on any dataset

1. Place your CSV in `data/` or pass a path.
2. Import the functions from `scripts/summary_statistics.py`.
3. Call `full_summary(your_series)` or the individual helpers.
4. Use `summary_by_group(df, value_col, group_col)` for segmented analysis.

See also the standalone template notebook and the `.py` runner script.
